In [6]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

In [7]:
data = pd.read_csv("heart.csv")

print("Dataset Shape:", data.shape)
data.head()

Dataset Shape: (920, 16)


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [8]:
data['target'] = data['num'].apply(lambda x: 1 if x > 0 else 0)

data[['num','target']].head()

,num,target
0,0,0
1,2,1
2,1,1
3,0,0
4,0,0


In [9]:
le = LabelEncoder()

for column in data.columns:
    if data[column].dtype == 'object':
        data[column] = le.fit_transform(data[column])

In [10]:
data['age'] = pd.cut(data['age'], bins=3, labels=[0,1,2]).cat.codes
data['trestbps'] = pd.cut(data['trestbps'], bins=3, labels=[0,1,2]).cat.codes
data['chol'] = pd.cut(data['chol'], bins=3, labels=[0,1,2]).cat.codes
data['thalch'] = pd.cut(data['thalch'], bins=3, labels=[0,1,2]).cat.codes

In [11]:
data.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num,target
0,1,2,1,0,3,2,1,1,0,1,0,2.3,0,0.0,0,0,0
1,2,2,1,0,0,2,1,0,0,1,1,1.5,1,3.0,1,2,1
2,3,2,1,0,0,1,1,0,0,1,1,2.6,1,2.0,2,1,1
3,4,0,1,0,2,1,1,0,1,2,0,3.5,0,0.0,1,0,0
4,5,0,0,0,1,1,1,0,0,2,0,1.4,2,0.0,1,0,0


In [12]:
data = data[['age','sex','cp','trestbps','chol','target']]

data.head()

,age,sex,cp,trestbps,chol,target
0,2,1,3,2,1,0
1,2,1,0,2,1,1
2,2,1,0,1,1,1
3,0,1,2,1,1,0
4,0,0,1,1,1,0


In [13]:
from pgmpy.models import DiscreteBayesianNetwork

model = DiscreteBayesianNetwork([
    ('age','target'),
    ('sex','target'),
    ('cp','target'),
    ('trestbps','target'),
    ('chol','target')
])

In [14]:
model.fit(data, estimator=MaximumLikelihoodEstimator)

print("Model trained successfully")

Model trained successfully


In [15]:
print("age values:", data['age'].unique())
print("sex values:", data['sex'].unique())
print("cp values:", data['cp'].unique())
print("trestbps values:", data['trestbps'].unique())
print("chol values:", data['chol'].unique())

age values: [2 0 1]
sex values: [1 0]
cp values: [3 0 2 1]
trestbps values: [ 2  1 -1  0]
chol values: [ 1  0  2 -1]


In [16]:
from pgmpy.inference import VariableElimination

inference = VariableElimination(model)

result = inference.query(
    variables=['target'],
    evidence={
        'age':1,
        'sex':1,
        'cp':0,
        'trestbps':1,
        'chol':1
    }
)

print(result)

+-----------+---------------+
| target    |   phi(target) |
+===========+===============+
| target(0) |        0.1892 |
+-----------+---------------+
| target(1) |        0.8108 |
+-----------+---------------+
